# Connect Four
### José Antonio Mérida Castejón - 201105
Para este laboratorio, vamos a estar implementando un agente capaz de jugar Connect Four implementando Minimax con Alpha-Beta pruning utilizando una heurística definida por nosotros. En este notebook, vamos a definir las clases y heurísticas para poder validar el funcionamiento y analizar de una manera más granular. Los agentes y código desarrollados acá serán exportados utilizando Pickle, para luego poder jugar en contra y ver demostraciones del funcionamiento utilizando una aplicación de streamlit.

## Task 2.1 - Implementación de Lógica de Juego y Algoritmo Base
Aquí vamos a definir la clase Connect4, dónde se maneja el estado del tablero, movimientos válidos, victorias, etc. Al igual que un algoritmo "base" de MiniMax para buscar la mejor jugada en una posición.

### Connect4
Al ser un ejercicio didáctico, la implementación cuenta con funciones entendibles / explícitas dónde el estado del tablero se representa por medio de un Array 2D. Las implementaciones más "formales" cuentan con una representación por medio de Bitboards y utilizan operaciones de bits con la finalidad de tener una mejor velocidad, pero se pierde un poco la lógica del juego. Hay algunas optimizaciones pequeñas para los algoritmos a implementar a futuro, estas se encuentran documentadas pero no es estrictamente necesario entenderlas para poder trabajar con esta clase.

In [12]:
import numpy as np

# Clase Connect4
class Connect4:
    # Inicializar estado del tablero
    def __init__(self, rows=6, cols=7):
        self.rows = rows
        self.cols = cols
        self.board = np.zeros((rows, cols), dtype=int)
        self.last_move = None # Almacena (row, col)

    # Obtener  movimientos válidos
    def get_valid_moves(self):
        '''Revisa que las columnas no estén llenas, está diseñado para revisar del centro
        hacia afuera. Esto ayuda al agente que implementa pruning, ya que las columnas
        más centrales son más importantes y es probable que encontremos un buen movimiento
        ahí.'''
        center = self.cols // 2
        moves = []
        for i in range(self.cols):
            col = center + (1 - 2 * (i % 2)) * (i + 1) // 2
            if self.board[0][col] == 0:
                moves.append(col)
        return moves

    # Realizar movimiento
    def make_move(self, col, player):
        # Revisa bottom-up para encontrar el espacio vacío más alto y lo llena
        for r in range(self.rows - 1, -1, -1):
            if self.board[r][col] == 0:
                self.board[r][col] = player
                self.last_move = (r, col)
                return r
        return None

    # Función para deshacer un movimiento, se utiliza en back-tracking
    def undo_move(self, col, row):
        self.board[row][col] = 0
        self.last_move = None

    # Verificar si un movimiento resultó en ganar
    def is_win_at(self, row, col, player):
        '''Para evitar escanear todas las posiciones, almacenamos la posición del último
        movimiento y limitamos nuestra búsqueda únicamente a las piezas que rodean este
        movimiento. Es imposible que se gane en otro lado de la tabla'''
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
        for dr, dc in directions:
            count = 1
            for delta in [1, -1]:
                r, c = row + delta * dr, col + delta * dc
                while 0 <= r < self.rows and 0 <= c < self.cols and self.board[r][c] == player:
                    count += 1
                    r += delta * dr
                    c += delta * dc
            if count >= 4:
                return True
        return False

    # Verificar si el juego terminó
    def is_terminal(self):
        # Verificar si el último movimiento resultó en ganar
        if self.last_move:
            r, c = self.last_move
            if self.is_win_at(r, c, self.board[r][c]):
                return self.board[r][c] # Retorna ID del jugador ganador
        # Si no hay ninguna casilla vacía es empate
        if not np.any(self.board[0] == 0):
            return 0
        return None

### Minimax
Aquí vamos a definir el `MinimaxAgent`, esta clase será utilizada como clase base para luego implementar el `AlphaBetaAgent`. De momento, la evaluación se bada únicamente en verificar si se ganó o no. 

In [ ]:
import math

# Agente que implementa Minimax para Connect 4
class MinimaxAgent:
    # Inicializar agente
    def __init__(self, player_id, depth=4):
        self.player_id = player_id
        self.opp_id = 1 if player_id == 2 else 2
        self.depth = depth
        self.nodes_visited = 0

    # Evaluar posición
    def evaluate(self, game):
        res = game.is_terminal() # Detección de estado terminal
        # Ganar (+100000), Perder (-100000), Empate (0)
        if res == self.player_id: return 100000
        if res == self.opp_id: return -100000
        return 0

    # Función para encontrar el mejor movimiento
    def get_best_move(self, game):
        # Inicializar búsqueda
        self.nodes_visited = 0
        best_val = -math.inf
        moves = game.get_valid_moves() # Obtener acciones posibles [cite: 43]
        best_move = moves[0] if moves else None
        
        # Bucle principal para evaluar cada columna posible
        for col in moves:
            # Realizar movimiento
            row = game.make_move(col, self.player_id)
            # Llamada inicial al algoritmo recursivo
            val = self._minimax(game, self.depth - 1, False)
            # Deshacer movimiento (Backtracking)
            game.undo_move(col, row)
            # Actualizar mejor jugada encontrada
            if val > best_val:
                best_val, best_move = val, col
        return best_move

    # Lógica recursiva de Minimax, toma como parámetro depth y si debemos minimizar o maximizar.
    def _minimax(self, game, depth, is_maximizing):
        self.nodes_visited += 1
        term = game.is_terminal()
        
        # Caso base: se alcanza el límite de profundidad o fin del juego
        if depth == 0 or term is not None:
            return self.evaluate(game)

        # Rama del Maximizador
        if is_maximizing:
            val = -math.inf
            # Probar cada acción válida
            for col in game.get_valid_moves():
                row = game.make_move(col, self.player_id)
                # Llamada recursiva: cambia al turno del oponente con depth = depth - 1
                # y maximizing = false
                val = max(val, self._minimax(game, depth - 1, False))
                game.undo_move(col, row)
            return val
        
        # Rama del Minimizador (El oponente)
        else:
            val = math.inf
            # Probar cada acción válida del oponente
            for col in game.get_valid_moves():
                row = game.make_move(col, self.opp_id)
                # Llamada recursiva: cambia al turno del agente con depth = depth - 1
                # y maximizing = true
                val = min(val, self._minimax(game, depth - 1, True))
                game.undo_move(col, row)
            return val

## Task 2.2 - Alpha Beta Pruning
Aquí vamos a implementar pruning Alpha-Beta, dónde modificamos la función recursiva para pasar y actualizar los valores $\alpha$ y $\beta$. Adicionalmente, vamos a realizar una demostración para mostrar la cantidad de nodos visitados con vs sin poda.

### Implementación AlphaBetaAgent
Aquí definimos el agente, utilizamos las funciones ya definidas en `MinimaxAgent` y únicamente modificamos la lógica de obtener el mejor movimiento utilizando la búsqueda implementada.

In [14]:
# Agente optimizado que hereda de MinimaxAgent para implementar Poda Alfa-Beta [cite: 48]
class AlphaBetaAgent(MinimaxAgent):
    # Función para encontrar el mejor movimiento utilizando límites alpha y beta
    def get_best_move(self, game):
        # Inicializar búsqueda y contadores
        self.nodes_visited = 0
        # alpha: mejor valor para MAX, beta: mejor valor para MIN
        best_val, alpha, beta = -math.inf, -math.inf, math.inf
        moves = game.get_valid_moves() 
        best_move = moves[0] if moves else None

        # Bucle principal para evaluar cada columna inicial
        for col in moves:
            # Realizar movimiento
            row = game.make_move(col, self.player_id)
            # Llamada inicial: se pasa alpha, beta, depth = depth - 1 y maximizing = false
            val = self._alpha_beta(game, self.depth - 1, alpha, beta, False)
            # Deshacer movimiento (Backtracking)
            game.undo_move(col, row)
            
            # Actualizar mejor jugada y el valor de alpha
            if val > best_val:
                best_val, best_move = val, col
            
            # El maximizador actualiza su límite inferior (alpha)
            alpha = max(alpha, best_val)
        return best_move

    # Lógica recursiva de Alfa-Beta, toma como parámetros alpha y beta para podar el árbol 
    def _alpha_beta(self, game, depth, alpha, beta, is_maximizing):
        self.nodes_visited += 1
        term = game.is_terminal()
        
        # Caso base: se alcanza el límite de profundidad o hay un estado terminal
        if depth == 0 or term is not None:
            return self.evaluate(game)

        # Rama del Maximizador
        if is_maximizing:
            val = -math.inf
            for col in game.get_valid_moves():
                row = game.make_move(col, self.player_id)
                # Llamada recursiva: pasa alpha/beta, depth = depth - 1 y maximizing = false
                val = max(val, self._alpha_beta(game, depth - 1, alpha, beta, False))
                game.undo_move(col, row)
                
                # Actualizar alpha y verificar si podemos podar la rama
                alpha = max(alpha, val)
                if alpha >= beta: 
                    break # Poda Alfa: el oponente ya tiene una mejor opción arriba 
            return val
        
        # Rama del Minimizador (El oponente)
        else:
            val = math.inf
            for col in game.get_valid_moves():
                row = game.make_move(col, self.opp_id)
                # Llamada recursiva: pasa alpha/beta, depth = depth - 1 y maximizing = true
                val = min(val, self._alpha_beta(game, depth - 1, alpha, beta, True))
                game.undo_move(col, row)
                
                # Actualizar beta y verificar si podemos podar la rama
                beta = min(beta, val)
                if alpha >= beta: 
                    break # Poda Beta: ya encontramos un movimiento peor que el garantizado arriba 
            return val

### Comparación Matemática de Rendimiento ($d=4$)

Para un factor de ramificación $b=7$ y una profundidad de búsqueda $d=4$, el número de nodos visitados se comporta de la siguiente manera:

#### Minimax Puro
Al no tener memoria de valores previos, Minimax es puramente exhaustivo y debe visitar cada nodo en cada nivel para asegurar que no se omita ninguna jugada.
* **Fórmula:** $\sum_{i=1}^{4} 7^i = 7^1 + 7^2 + 7^3 + 7^4$
* **Cálculo:** $7 + 49 + 343 + 2,401 = \mathbf{2,800 \text{ nodos}}$

#### 2. Poda Alfa-Beta (Caso Óptimo por Equivalencia)
Cuando todas las hojas devuelven $0$, el algoritmo alcanza su complejidad óptima de $O(b^{d/2})$. Una vez que se establece un "benchmark" de 0 en la primera rama, el resto se poda agresivamente:

* **Rama de Referencia (Columna 3):** Es la primera en explorarse. Al no tener un valor de $\alpha$ previo, debe profundizar más para establecer el primer 0.
    * **Primer sub-árbol ($d=2$):** Para que el primer nodo en $d=2$ devuelva un valor, su primer hijo en $d=1$ debe revisar sus **7 hojas** para encontrar el primer 0 ($1 + 7 = 8$ nodos). Luego, los otros 6 hijos en $d=1$ solo revisan **una hoja** cada uno para confirmar el $0 \geq 0$ ($6 \times 2 = 12$ nodos). Sumando el nodo raíz de este nivel, el primer sub-árbol suma **21 nodos**.
    * **Hermanos en $d=3$:** La raíz de la columna ($d=3$) ya tiene $\beta=0$. Ahora revisa sus otros 6 hijos en $d=2$. Cada hijo entra ($1$ nodo), revisa su primer descendiente en $d=1$ (el cual revisa sus **7 hojas** para establecer $\alpha=0$, sumando 8 nodos). En ese instante, se cumple $\alpha(0) \geq \beta(0)$ y se poda el resto de la rama.
    * **Costo por hermano:** $6 \text{ hijos} \times (1 \text{ en } d=2 + 8 \text{ en } d=1) = \mathbf{54 \text{ nodos}}$.
    * **Costo total Columna 3:** $1 (\text{raíz } d=3) + 21 + 54 = \mathbf{76 \text{ nodos}}$.

* **Ramas de Prueba (Restantes 6 columnas):** Con $\alpha = 0$ ya establecido en la raíz global ($d=4$), el algoritmo solo necesita la "prueba mínima" de que estas ramas no son mejores.
    * Cada columna entra al nivel $d=3$, revisa su primer hijo en $d=2$, el cual revisa sus 7 descendientes en $d=1$. Cada descendiente en $d=1$ toca **una sola hoja**, devuelve 0, y activa la poda inmediata por la condición $\alpha(0) \geq \beta(0)$.
    * **Estructura:** $1 (\text{nodo } d=3) + 1 (\text{nodo } d=2) + (7 \text{ hijos en } d=1 \times 2 \text{ nodos c/u}) = \mathbf{16 \text{ nodos}}$.
    * **Costo total:** $6 \text{ ramas} \times 16 \text{ nodos} = \mathbf{96 \text{ nodos}}$.

**Total:** $76 + 96 = \mathbf{172 \text{ nodos}}$


In [15]:
env = Connect4()
d = 4 # Probando con d=4 [cite: 46]

agent_pure = MinimaxAgent(player_id=1, depth=d)
agent_pure.get_best_move(env)

agent_ab = AlphaBetaAgent(player_id=1, depth=d)
agent_ab.get_best_move(env)

print(f"Nodes visited (Pure Minimax): {agent_pure.nodes_visited}")
print(f"Nodes visited (Alpha-Beta): {agent_ab.nodes_visited}")
print(f"Reduction: {((agent_pure.nodes_visited - agent_ab.nodes_visited) / agent_pure.nodes_visited) * 100:.2f}%")

Nodes visited (Pure Minimax): 2800
Nodes visited (Alpha-Beta): 172
Reduction: 93.86%


#### Conclusión
El análisis teórico es correcto y la reducción del **93.86%** demuestra que Alpha-Beta transforma el crecimiento exponencial de $O(b^d)$ en una búsqueda mucho más eficiente de $O(b^{d/2})$. Esto permite que el agente ignore más del 90% del árbol sin perder la capacidad de tomar la decisión óptima, demostrando ser una necesidad absoluta para juegos con factores de ramificación elevados. Adicionalmente, logramos validar el comportamiento interno de ambos de nuestros agentes.

## Task 2.3 - Heurística de Evaluación
Para esta heurística, nos basamos en las siguientes observaciones sobre el juego y su implementación algorítmica:

* **Control del Centro:** La columna central es el punto más crítico del tablero porque participa en la mayor cantidad de líneas ganadoras posibles. Se implementa un bono estático inicial iterando únicamente sobre la columna central y multiplicando la cantidad de fichas del agente por un peso positivo.
* **Prioridad Defensiva (Paranoia Asimétrica):** En juegos de suma cero, evitar perder es más urgente que ganar. Dado que Minimax asume juego perfecto del oponente, la heurística castiga las amenazas enemigas con mayor severidad de lo que premia las propias (ej. $-80$ por una amenaza rival vs $+50$ por una propia), forzando al agente a priorizar el bloqueo.
* **Recorrido del Tablero (Sliding Window):** Para evaluar el estado del juego, el algoritmo escanea las 69 posibles líneas de victoria en un tablero de 6x7. Extrae "ventanas" (sub-arreglos) de exactamente 4 casillas de longitud, barriendo en dirección horizontal, vertical y en ambas diagonales.
* **Detección de Rutas Bloqueadas:**  Una línea de 3 fichas es inútil si la cuarta casilla pertenece al oponente o choca con el borde. Esto se detecta automáticamente por exclusión: se exige que la ventana tenga 3 fichas del jugador y exactamente 1 espacio vacío (`window.count(0) == 1`). Si hay una ficha enemiga, el conteo de vacíos es 0, la condición falla y la ruta se descarta sumando 0 puntos, entendiendo que matemáticamente está muerta.
* **Amenazas Dobles (Forks por Traslape):**  El valor de las "trampas" surge de forma aditiva. Al evaluar cada ventana de 4 de forma independiente, una ficha colocada en una intersección estratégica pertenecerá a múltiples ventanas simultáneamente. Si un movimiento crea un 3-en-raya horizontal y uno diagonal, la función sumará los puntajes de ambas ventanas. Minimax seleccionará naturalmente esta ruta al maximizar el puntaje, priorizando jugadas que generen múltiples ejes de ataque imposibles de bloquear a la vez.
* **Normalización de Estados Terminales:** Para evitar que el agente prefiera un tablero con múltiples amenazas inconclusas sobre una victoria real, los puntajes terminales actúan como cotas absolutas. El tablero tiene un máximo de 69 ventanas; en el límite superior teórico de que todas valieran $+50$, la heurística sumaría $3,450$. Por lo tanto, el puntaje de victoria se define en $+100,000$ (y la derrota en $-100,000$), garantizando que ninguna suma heurística intermedia pueda superar matemáticamente el valor de ganar o perder el juego.

In [16]:
def strategic_evaluate(self, game):
    # 1. Terminal State Check (Rules-based)
    res = game.is_terminal()
    if res == self.player_id: return 1000   # Win [cite: 40]
    if res == self.opp_id: return -1000    # Loss [cite: 40]
    if res == 0: return 0                  # Draw [cite: 40]

    score = 0
    
    # 2. Center Column Preference 
    # The center is part of the most possible winning lines
    center_col = game.cols // 2
    center_array = [int(i) for i in list(game.board[:, center_col])]
    score += center_array.count(self.player_id) * 3

    # 3. Sliding Window Scoring
    # We check all possible horizontal, vertical, and diagonal lines
    def evaluate_window(window, player, opponent):
        win_score = 0
        if window.count(player) == 3 and window.count(0) == 1:
            win_score += 10  # Reward 3-in-a-row 
        elif window.count(player) == 2 and window.count(0) == 2:
            win_score += 2
            
        if window.count(opponent) == 3 and window.count(0) == 1:
            win_score -= 8   # Penalize opponent's 3-in-a-row (Defensive play)
            
        return win_score

    # Horizontal Score
    for r in range(game.rows):
        row_array = [int(i) for i in list(game.board[r,:])]
        for c in range(game.cols - 3):
            window = row_array[c:c+4]
            score += evaluate_window(window, self.player_id, self.opp_id)

    # Vertical Score
    for c in range(game.cols):
        col_array = [int(i) for i in list(game.board[:,c])]
        for r in range(game.rows - 3):
            window = col_array[r:r+4]
            score += evaluate_window(window, self.player_id, self.opp_id)

    # Diagonal Scores
    for r in range(game.rows - 3):
        for c in range(game.cols - 3):
            # Positive slope
            window = [game.board[r+i][c+i] for i in range(4)]
            score += evaluate_window(window, self.player_id, self.opp_id)
            # Negative slope
            window = [game.board[r+3-i][c+i] for i in range(4)]
            score += evaluate_window(window, self.player_id, self.opp_id)

    return score

# Inject the new heuristic into the existing Agent classes
MinimaxAgent.evaluate = strategic_evaluate

In [17]:
env = Connect4()
d = 4 # Probando con d=4 [cite: 46]

agent_pure = MinimaxAgent(player_id=1, depth=d)
agent_pure.get_best_move(env)

agent_ab = AlphaBetaAgent(player_id=1, depth=d)
agent_ab.get_best_move(env)

print(f"Nodes visited (Pure Minimax): {agent_pure.nodes_visited}")
print(f"Nodes visited (Alpha-Beta): {agent_ab.nodes_visited}")
print(f"Reduction: {((agent_pure.nodes_visited - agent_ab.nodes_visited) / agent_pure.nodes_visited) * 100:.2f}%")

Nodes visited (Pure Minimax): 2800
Nodes visited (Alpha-Beta): 177
Reduction: 93.68%


## Export Agent for Streamlit
Aquí exportamos el agente entrenado para utilizarlo en la aplicación de Streamlit.

In [ ]:
import pickle
import sys
sys.path.insert(0, ".")

# The agent classes are now in notebook_agents.py
# This makes them importable when unpickling
from notebook_agents import AlphaBetaAgent

# Create the agent with strategic heuristic (depth=5 for good performance)
trained_agent = AlphaBetaAgent(player_id=2, depth=5)

# Export to pickle file
with open("trained_agent.pkl", "wb") as f:
    pickle.dump(trained_agent, f)

print("✓ Agent exported successfully to trained_agent.pkl")
print(f"✓ Agent configuration: depth={trained_agent.depth}, player_id={trained_agent.player_id}")
print("
Now run: streamlit run app.py")